# 기계학습 실습 12: WikiText-2로 배우는 Masked Language Modeling

**Machine Learning and Bioinformatics (MLBI) Lab, Dankook University**

영어 Wikipedia 텍스트의 일부 토큰을 가리고, 양쪽 문맥으로 원래 토큰을 복원하는 작은 Transformer encoder를 학습합니다. 앞의 masked image reconstruction 실습과 같은 자기지도학습 원리입니다.

### 학습 목표
1. 단어와 서브워드 토큰의 차이를 확인한다.
2. 입력 토큰, 위치 임베딩, self-attention, vocabulary 출력층의 역할을 이해한다.
3. 가린 토큰에서만 loss를 계산하여 모델을 **처음부터 학습**한다.
4. 고정된 검증 마스크로 학습 전후 loss, top-1/top-5 정확도를 비교한다.
5. 직접 입력한 문장의 `[MASK]` 후보를 확인하고 모델을 저장·복원한다.

**토크나이저만 사전학습된 BERT WordPiece를 사용합니다. 아래 TinyMLM의 가중치는 무작위로 초기화하며 BERT 가중치를 가져오지 않습니다.** 작은 데이터와 짧은 학습은 원리 확인용이므로 자연스러운 빈칸 예측을 보장하지 않습니다. 마지막에는 사전학습 모델을 선택적으로 비교할 수 있습니다.

### 실행 안내
- Google Colab의 **런타임 → 런타임 유형 변경 → GPU**를 권장합니다. GPU가 없어도 실행되지만 느릴 수 있습니다.
- 기본값은 빠른 실습용 부분 데이터입니다. 실제 속도를 첫 20 step에서 측정합니다. 5~10분 완료를 보장하는 설정은 아닙니다.
- 최초 실행에는 데이터셋·토크나이저 다운로드를 위한 인터넷이 필요합니다. 유료 API나 API 키는 필요하지 않습니다.
- 셀을 위에서 아래로 실행하세요. Colab 세션 종료 전에 `outputs/`의 필요한 파일을 다운로드하거나 Drive에 복사하세요.

## 0. 패키지와 실습 설정

PyTorch로 모델을 직접 작성하고, Hugging Face `datasets`와 `transformers`는 데이터 로딩과 토큰화에 사용합니다. 설치 후 이미 불러온 패키지의 버전 충돌이 나타나면 런타임을 재시작하세요.

In [ ]:
%pip install -q "torch>=2.3,<3" "transformers>=4.40,<6" "datasets>=3,<5" "pyarrow>=21,<26" pandas matplotlib

In [ ]:
import json
import random
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from IPython.display import display

In [ ]:
SEED = 42
QUICK_RUN = True
TOKENIZER_NAME = "google-bert/bert-base-uncased"
BLOCK_SIZE = 128
MASK_PROB = 0.15
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
FF_DIM = 512
DROPOUT = 0.1
EPOCHS = 20 if QUICK_RUN else 50
MAX_TRAIN_BLOCKS = 3000 if QUICK_RUN else None
MAX_VALID_BLOCKS = 256 if QUICK_RUN else None
MAX_TEST_BLOCKS = 256 if QUICK_RUN else None
LEARNING_RATE = 3e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 8
USE_AMP = DEVICE.type == "cuda"
OUTPUT_DIR = Path("outputs/wikitext2_tiny_mlm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert D_MODEL % N_HEADS == 0
assert BLOCK_SIZE >= 4 and 0 < MASK_PROB < 1
print("PyTorch:", torch.__version__, "device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("batch size:", BATCH_SIZE, "epochs:", EPOCHS)

## 1. WikiText-2 데이터 살펴보기

`Salesforce/wikitext`의 `wikitext-2-raw-v1`을 사용합니다. 원본에는 학습 36,718행, 검증 3,760행, 테스트 4,358행이 있습니다. **행은 문장이 아닙니다.** 빈 줄·제목·여러 문장이 있는 문단이 섞여 있습니다.

아래에서는 빈 줄과 제목을 제외합니다. 원래 split을 유지하며, 테스트 데이터는 학습이나 모델 선택에 사용하지 않습니다. 정확한 문장 수를 세는 작업은 수행하지 않습니다.

출처: [Salesforce WikiText 데이터셋](https://huggingface.co/datasets/Salesforce/wikitext), [원 논문](https://arxiv.org/abs/1609.07843). 원문 데이터를 재배포하려면 데이터셋 카드와 원 Wikipedia 문서의 라이선스를 확인하세요.

In [ ]:
raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def clean_rows(dataset):
    return [text.strip() for text in dataset["text"]
            if text.strip() and not text.strip().startswith("=")]

texts = {split: clean_rows(raw[split]) for split in ("train", "validation", "test")}
display(pd.DataFrame([
    {"split": split, "raw_rows": len(raw[split]), "retained_text_rows": len(texts[split])}
    for split in texts
]))
print(texts["train"][0][:800])

## 2. 문장 → 서브워드 토큰 → 정수 ID

WordPiece는 자주 쓰는 단어를 하나의 토큰으로, 드문 단어를 여러 조각으로 표현할 수 있습니다. `##`는 단어 중간에 이어지는 조각을 표시합니다. 이 토크나이저는 소문자화를 수행합니다.

`[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`는 특별한 역할을 가진 토큰입니다. 이번 실습은 **토큰 단위 마스킹**이며, 단어의 모든 조각을 동시에 가리는 whole-word masking은 아닙니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
example_text = "The researchers discussed the incomprehensibility of the unfamiliar terminology."
encoded = tokenizer(example_text)
display(pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    "token_id": encoded["input_ids"],
}))
print("Vocabulary size:", len(tokenizer))
print("Special tokens:", tokenizer.special_tokens_map)

## 3. 길이 128의 입력 블록 만들기

각 split 안에서 텍스트를 토큰화하고 문단 사이에 `[SEP]`를 넣은 뒤, 고정 길이 블록으로 묶습니다. 서로 다른 split은 합치지 않습니다. 한 블록은 여러 문장의 일부를 포함할 수 있으며, 단순한 packing 때문에 문서 경계를 넘을 수도 있습니다.

각 블록의 처음과 끝에는 `[CLS]`, `[SEP]`를 넣고 마지막 불완전 블록은 버립니다. 따라서 아래 **블록 수가 실제 학습 샘플 수**입니다. 빠른 실습에서는 split별로 고정된 seed를 사용해 블록을 추출합니다.

In [ ]:
def make_blocks(rows, max_blocks, seed):
    stream = []
    for start in range(0, len(rows), 256):
        batch = tokenizer(rows[start:start + 256], add_special_tokens=False,
                          truncation=False, verbose=False)["input_ids"]
        for ids in batch:
            stream.extend(ids)
            stream.append(tokenizer.sep_token_id)
    width = BLOCK_SIZE - 2
    n_full = len(stream) // width
    if n_full == 0:
        raise ValueError("텍스트가 너무 적어 입력 블록을 만들 수 없습니다.")
    indices = np.arange(n_full)
    if max_blocks is not None and n_full > max_blocks:
        indices = np.random.default_rng(seed).choice(n_full, max_blocks, replace=False)
    blocks = torch.tensor([
        [tokenizer.cls_token_id] + stream[int(i)*width:(int(i)+1)*width]
        + [tokenizer.sep_token_id] for i in indices
    ], dtype=torch.long)
    return blocks, len(stream), n_full

train_ids, train_tokens, train_full = make_blocks(texts["train"], MAX_TRAIN_BLOCKS, SEED)
valid_ids, valid_tokens, valid_full = make_blocks(texts["validation"], MAX_VALID_BLOCKS, SEED + 1)
test_ids, test_tokens, test_full = make_blocks(texts["test"], MAX_TEST_BLOCKS, SEED + 2)
display(pd.DataFrame([
    {"split": name, "tokens_including_separators": tokens,
     "available_blocks": full, "used_blocks": len(ids)}
    for name, ids, tokens, full in [
        ("train", train_ids, train_tokens, train_full),
        ("validation", valid_ids, valid_tokens, valid_full),
        ("test", test_ids, test_tokens, test_full),
    ]
]))
print("One training block:\n", tokenizer.decode(train_ids[0]))

## 4. 마스크와 학습 정답 만들기

특수 토큰을 제외한 위치 중 약 15%를 선택합니다. 원리를 명확히 보기 위해 **선택된 토큰을 모두 `[MASK]`로 바꾸는 교육용 방식**을 사용합니다. BERT 원 논문의 80% mask / 10% random / 10% unchanged 전략과는 다릅니다.

- 정답은 가리기 전 토큰 ID입니다.
- 선택하지 않은 위치의 정답은 `-100`으로 두고 loss에서 제외합니다.
- 학습 마스크는 배치마다 새로 만듭니다.
- 검증·테스트 마스크는 한 번 생성하여 고정합니다. 테스트 평가는 마지막에 한 번 수행합니다.
- 최소 한 개의 유효 토큰을 선택하여 빈 loss를 방지합니다.

In [ ]:
special_ids = torch.tensor(tokenizer.all_special_ids)

def mask_tokens(ids, generator=None):
    eligible = ~torch.isin(ids, special_ids)
    selected = (torch.rand(ids.shape, generator=generator) < MASK_PROB) & eligible
    for row in range(len(ids)):
        choices = eligible[row].nonzero(as_tuple=True)[0]
        if len(choices) == 0:
            raise ValueError("특수 토큰만 있는 블록입니다.")
        if not selected[row].any():
            pick = torch.randint(len(choices), (1,), generator=generator).item()
            selected[row, choices[pick]] = True
    labels = ids.clone()
    labels[~selected] = -100
    masked = ids.clone()
    masked[selected] = tokenizer.mask_token_id
    return masked, labels

valid_inputs, valid_labels = mask_tokens(valid_ids, torch.Generator().manual_seed(SEED + 10))
test_inputs, test_labels = mask_tokens(test_ids, torch.Generator().manual_seed(SEED + 20))
print("Original:", tokenizer.decode(valid_ids[0]))
print("Masked:  ", tokenizer.decode(valid_inputs[0]))
positions = (valid_labels[0] != -100).nonzero(as_tuple=True)[0]
display(pd.DataFrame({
    "position": positions.tolist(),
    "target_token": tokenizer.convert_ids_to_tokens(valid_labels[0, positions].tolist()),
}))
assert torch.all(valid_labels[torch.isin(valid_ids, special_ids)] == -100)

train_loader = DataLoader(TensorDataset(train_ids), batch_size=BATCH_SIZE,
                          shuffle=True, generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(TensorDataset(valid_inputs, valid_labels), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(test_inputs, test_labels), batch_size=BATCH_SIZE)

## 5. 작은 Transformer encoder 구성

`토큰 임베딩 + 위치 임베딩 → Transformer encoder → LayerNorm → 어휘별 점수`

각 위치에서 왼쪽과 오른쪽 문맥을 모두 볼 수 있도록 causal mask를 사용하지 않습니다. 단, `[PAD]` 위치는 attention에서 제외합니다. 모델은 **가린 위치의 은닉 벡터만 어휘 출력층에 전달**하여 계산량과 메모리를 줄입니다.

출력층과 입력 임베딩의 가중치를 공유합니다(weight tying). 각 encoder layer는 별도로 생성하여 독립적으로 초기화합니다.

In [ ]:
class TinyMLM(nn.Module):
    def __init__(self, vocab_size, max_length, d_model, n_heads, n_layers,
                 ff_dim, dropout, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.input_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, ff_dim, dropout,
                                       activation="gelu", batch_first=True, norm_first=True)
            for _ in range(n_layers)
        ])
        self.output_norm = nn.LayerNorm(d_model)
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[pad_id].zero_()

    def forward(self, input_ids, predict_positions):
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        h = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        h = self.dropout(self.input_norm(h))
        for layer in self.layers:
            h = layer(h, src_key_padding_mask=input_ids.eq(self.pad_id))
        h = self.output_norm(h)[predict_positions]
        return F.linear(h, self.token_embedding.weight, self.output_bias)

model_config = dict(vocab_size=len(tokenizer), max_length=BLOCK_SIZE,
                    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                    ff_dim=FF_DIM, dropout=DROPOUT, pad_id=tokenizer.pad_token_id)
model = TinyMLM(**model_config).to(DEVICE)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

def amp_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()

@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    loss_sum = correct1 = correct5 = n_tokens = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        selected = labels.ne(-100)
        targets = labels[selected]
        with amp_context():
            logits = net(inputs, selected)
            loss = F.cross_entropy(logits, targets, reduction="sum")
        top5 = logits.topk(5, dim=-1).indices
        loss_sum += loss.item()
        correct1 += (top5[:, 0] == targets).sum().item()
        correct5 += (top5 == targets[:, None]).any(dim=1).sum().item()
        n_tokens += targets.numel()
    return {"loss": loss_sum / n_tokens, "top1": correct1 / n_tokens,
            "top5": correct5 / n_tokens, "masked_tokens": n_tokens}

before_metrics = evaluate(model, valid_loader)
print("Before training (validation):", before_metrics)

## 6. 모델 학습하기

AdamW, gradient clipping, GPU 혼합 정밀도를 사용합니다. loss는 각 배치의 **선택된 토큰 수**로 가중 평균합니다. 첫 최대 20 step의 속도로 학습 루프 시간을 추정하며, 검증·저장 시간은 이 추정에 포함하지 않습니다.

매 epoch 검증 loss가 가장 낮은 가중치를 메모리에 보관합니다. 검증 마스크는 동일하지만 학습 마스크는 계속 달라지므로, 두 loss가 완전히 같은 조건에서 측정된 것은 아닙니다.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
history = []
best_loss = before_metrics["loss"]
best_epoch = 0
best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
training_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = token_count = 0
    epoch_start = time.perf_counter()
    for step, (ids,) in enumerate(train_loader, 1):
        inputs, labels = mask_tokens(ids)
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        selected = labels.ne(-100)
        targets = labels[selected]
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            logits = model(inputs, selected)
            loss = F.cross_entropy(logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * targets.numel()
        token_count += targets.numel()
        if epoch == 1 and step == min(20, len(train_loader)):
            seconds_per_step = (time.perf_counter() - epoch_start) / step
            estimate = seconds_per_step * len(train_loader) * EPOCHS / 60
            print(f"Measured: {seconds_per_step:.3f} sec/step; estimated training loops: {estimate:.1f} min (excluding validation/save)")
    val = evaluate(model, valid_loader)
    row = dict(epoch=epoch, train_loss=loss_sum / token_count,
               val_loss=val["loss"], val_top1=val["top1"], val_top5=val["top5"],
               seconds=time.perf_counter() - epoch_start)
    history.append(row)
    if val["loss"] < best_loss:
        best_loss, best_epoch = val["loss"], epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch}/{EPOCHS}: train={row['train_loss']:.4f}, val={row['val_loss']:.4f}, "
          f"top1={row['val_top1']:.3%}, top5={row['val_top5']:.3%}, {row['seconds']:.1f}s")

model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}; total elapsed: {(time.perf_counter()-training_start)/60:.1f} min")
history_df = pd.DataFrame(history)
display(history_df)

## 7. 학습 곡선과 최종 평가

Top-1은 가장 높은 점수의 토큰이 원래 토큰과 같은 비율, top-5는 상위 5개 후보 안에 원래 토큰이 있는 비율입니다. **마스킹된 위치만** 평가하며 단어 전체의 정확도가 아닙니다. 빈칸에는 의미상 타당한 다른 답도 가능하지만 이 평가는 원문 토큰만 정답으로 봅니다.

일반적인 다음 토큰 언어 모델의 perplexity와 혼동하지 않도록 여기서는 MLM loss와 정확도를 보고합니다. 빠른 실습의 테스트 결과는 전체 테스트 split이 아닌 선택된 블록에 대한 값입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df.epoch, history_df.train_loss, marker="o", label="Train")
axes[0].plot([0] + history_df.epoch.tolist(),
             [before_metrics["loss"]] + history_df.val_loss.tolist(), marker="o", label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Masked-token cross entropy")
axes[1].plot([0] + history_df.epoch.tolist(),
             [before_metrics["top1"]] + history_df.val_top1.tolist(), marker="o", label="Top-1")
axes[1].plot([0] + history_df.epoch.tolist(),
             [before_metrics["top5"]] + history_df.val_top5.tolist(), marker="o", label="Top-5")
axes[1].set(xlabel="Epoch", ylabel="Validation masked-token accuracy")
for ax in axes:
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()

after_metrics = evaluate(model, valid_loader)
# Test is evaluated only after model selection by validation loss.
final_test_metrics = evaluate(model, test_loader)
display(pd.DataFrame([before_metrics, after_metrics, final_test_metrics],
                     index=["validation_before", "validation_best", "test_final"]))

## 8. 같은 검증 문장에서 복원 결과 보기

검증 데이터의 고정된 마스크와 원래 정답, 학습한 모델의 후보를 나란히 봅니다. 한 번에 여러 토큰이 가려진 경우 각 위치의 예측은 다른 가려진 위치가 여전히 `[MASK]`인 상태에서 계산됩니다.

In [ ]:
@torch.no_grad()
def show_reconstruction(index=0):
    model.eval()
    inputs = valid_inputs[index:index+1].to(DEVICE)
    labels = valid_labels[index:index+1].to(DEVICE)
    selected = labels.ne(-100)
    with amp_context():
        logits = model(inputs, selected)
    candidates = logits.topk(5, dim=-1).indices.cpu()
    positions = selected[0].nonzero(as_tuple=True)[0].cpu()
    predicted = inputs[0].cpu().clone()
    predicted[positions] = candidates[:, 0]
    print("Original:", tokenizer.decode(valid_ids[index]))
    print("Masked:  ", tokenizer.decode(inputs[0].cpu()))
    print("Predicted:", tokenizer.decode(predicted))
    return pd.DataFrame([
        {"position": pos, "answer": tokenizer.convert_ids_to_tokens(int(valid_ids[index, pos])),
         "top_5": tokenizer.convert_ids_to_tokens(top.tolist())}
        for pos, top in zip(positions.tolist(), candidates)
    ])

display(show_reconstruction(0))

## 9. 직접 입력한 문장의 빈칸 예측

영어 문장에 `[MASK]`를 넣어 보세요. 하나의 `[MASK]`는 **서브워드 토큰 하나**를 예측합니다. 아래 문장은 사용자가 입력하는 시연용 예문이며 데이터셋 문장을 인용한 것이 아닙니다. 작은 모델이 기능어나 자주 나오는 단어에 편중되는지도 관찰하세요.

In [ ]:
@torch.no_grad()
def predict_masks(text, net=model, top_k=5):
    net.eval()
    ids = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"]
    if ids.shape[1] > BLOCK_SIZE:
        raise ValueError(f"입력을 특수 토큰 포함 {BLOCK_SIZE}토큰 이하로 줄여 주세요.")
    ids = ids.to(DEVICE)
    selected = ids.eq(tokenizer.mask_token_id)
    if not selected.any():
        raise ValueError("문장에 [MASK]를 하나 이상 넣어 주세요.")
    with amp_context():
        logits = net(ids, selected)
    probs = logits.float().softmax(dim=-1)
    values, indices = probs.topk(min(top_k, len(tokenizer)), dim=-1)
    rows = []
    for mask_no, (scores, tokens) in enumerate(zip(values.cpu(), indices.cpu()), 1):
        for rank, (score, token) in enumerate(zip(scores, tokens), 1):
            rows.append({"mask": mask_no, "rank": rank,
                         "token": tokenizer.convert_ids_to_tokens(token.item()),
                         "probability": score.item()})
    return pd.DataFrame(rows)

user_text = "The scientist studied the [MASK] in the laboratory."
display(predict_masks(user_text))

## 10. 모델과 토크나이저 저장·복원

가장 낮은 검증 loss의 모델을 저장합니다. 구성과 가중치, 토크나이저, 학습 이력과 평가 결과를 함께 보관합니다. 이 checkpoint는 아래 TinyMLM 클래스로 읽는 형식이며 Hugging Face 사전학습 모델 형식은 아닙니다.

In [ ]:
torch.save({"model_config": model_config, "state_dict": best_state,
            "best_epoch": best_epoch}, OUTPUT_DIR / "tiny_mlm.pt")
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")
history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
(OUTPUT_DIR / "metrics.json").write_text(json.dumps({
    "seed": SEED, "dataset": "Salesforce/wikitext", "subset": "wikitext-2-raw-v1",
    "mask_probability": MASK_PROB, "quick_run": QUICK_RUN,
    "blocks": {"train": len(train_ids), "validation": len(valid_ids), "test": len(test_ids)},
    "validation_before": before_metrics, "validation_best": after_metrics,
    "test_final": final_test_metrics,
}, indent=2), encoding="utf-8")

checkpoint = torch.load(OUTPUT_DIR / "tiny_mlm.pt", map_location="cpu", weights_only=True)
restored_model = TinyMLM(**checkpoint["model_config"]).to(DEVICE)
restored_model.load_state_dict(checkpoint["state_dict"])
# The saved tokenizer can be loaded in a new session using this path.
restored_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR / "tokenizer")
assert restored_tokenizer.get_vocab() == tokenizer.get_vocab()
display(predict_masks(user_text, net=restored_model))
print("Saved:", OUTPUT_DIR.resolve())

## 11. 선택: 사전학습 BERT와 비교

`RUN_PRETRAINED_COMPARISON=True`로 바꾸면 사전학습 BERT 가중치를 추가로 다운로드합니다. 데이터량·모델 크기·학습량이 다르므로 공정한 성능 대결이 아니라 **교육용 소형 모델과 대규모 사전학습의 차이**를 보는 시연입니다. 다운로드 용량과 메모리 사용량이 늘어납니다.

In [ ]:
RUN_PRETRAINED_COMPARISON = False
if RUN_PRETRAINED_COMPARISON:
    from transformers import AutoModelForMaskedLM
    pretrained = AutoModelForMaskedLM.from_pretrained(TOKENIZER_NAME).to(DEVICE).eval()
    inputs = tokenizer(user_text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = pretrained(**inputs).logits[inputs["input_ids"].eq(tokenizer.mask_token_id)]
    probs, ids = logits.float().softmax(-1).topk(5, dim=-1)
    display(pd.DataFrame([
        {"mask": i + 1, "token": tokenizer.convert_ids_to_tokens(token.item()),
         "probability": prob.item()}
        for i, (row_probs, row_ids) in enumerate(zip(probs.cpu(), ids.cpu()))
        for prob, token in zip(row_probs, row_ids)
    ]))
    del pretrained
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("Optional pretrained comparison skipped.")

## 12. 직접 바꿔볼 것

1. `MASK_PROB`를 0.15에서 0.30으로 바꾸면 학습 난이도와 남은 문맥은 어떻게 달라질까요?
2. `N_LAYERS`, `D_MODEL`, `MAX_TRAIN_BLOCKS`를 바꾸어 속도와 검증 성능을 비교하세요.
3. 마스킹 위치를 단어 단위로 묶으면 부분 단어를 보고 답을 맞히는 경우가 어떻게 달라질까요?
4. 입력에서 오른쪽 문맥을 지우면 예측 후보가 어떻게 바뀌나요?
5. 토큰 accuracy가 높아도 사람이 보기에 자연스럽지 않을 수 있는 이유는 무엇인가요?

설정을 비교할 때 같은 검증 블록과 마스크를 사용하세요. 마스크 비율을 바꾸는 실험은 평가 난이도도 바뀌므로 직접 비교에 주의합니다. 반복적인 모델 선택은 검증 데이터로 하고 테스트는 최종 평가용으로 남겨 둡니다.

### 문제 해결
- **GPU 메모리 부족:** `BATCH_SIZE`부터 줄이고 런타임을 재시작합니다.
- **CPU에서 너무 느림:** `MAX_TRAIN_BLOCKS=128`, `MAX_VALID_BLOCKS=32`, `MAX_TEST_BLOCKS=32`, `EPOCHS=1`로 동작만 확인합니다.
- **다운로드 실패:** 네트워크 연결과 Hugging Face 접근 가능 여부를 확인합니다.
- **예측이 부자연스러움:** 짧은 학습과 작은 모델의 한계입니다. 과적합 여부를 확인하면서 데이터량·epoch를 늘리고 선택 실습의 사전학습 결과와 비교합니다.

### 참고 자료
- [WikiText-2 데이터셋](https://huggingface.co/datasets/Salesforce/wikitext)
- [BERT 논문: masked language modeling](https://arxiv.org/abs/1810.04805)
- [Hugging Face MLM 안내](https://huggingface.co/docs/transformers/tasks/masked_language_modeling)
- [PyTorch TransformerEncoderLayer](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html)